# Reviewer A3b — True-State Coverage, Bayes Floor, and Readout Difficulty

Focused follow-up to Reviewer 2, Comment 6.

For k in {3,4}, V=8:
- training sizes: 12k, 24k, 48k, 96k,
- exact one-hot true Markov state as input,
- linear, MLP, wider/deeper MLP next-token readouts,
- nonparametric state→next-token lookup estimator,
- exact Bayes floor H*,
- unique-state and stationary-mass coverage,
- five source seeds.

No Qwen inference and no Transformer retraining.

In [ ]:
import os, sys, json, math, random, gc, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASE=Path.cwd()/ "Markov_Context_Paper"
OUT=BASE/"Reviewer_A3b_TrueState_Coverage"
CSV_DIR=OUT/"csv"; FIG_DIR=OUT/"figures"; RUN_DIR=OUT/"runs"
for p in [OUT,CSV_DIR,FIG_DIR,RUN_DIR]: p.mkdir(parents=True,exist_ok=True)

V=8
KS=[3,4]
SEEDS=[0,1,2,3,4]
TRAIN_SIZES=[12_000,24_000,48_000,96_000]
N_VAL=8_000
N_TEST=20_000
PROBE_BATCH=1024
PATIENCE=5
LINEAR_EPOCHS=30
MLP_EPOCHS=40
LINEAR_LR=1e-3
MLP_LR=5e-4
WEIGHT_DECAY=1e-4

print("Device:",DEVICE)
if torch.cuda.is_available(): print("GPU:",torch.cuda.get_device_name(0))
print("Output:",OUT)


In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def save_json(path,obj):
    path.parent.mkdir(parents=True,exist_ok=True)
    with open(path,"w",encoding="utf-8") as f:
        json.dump(obj,f,indent=2,default=lambda x: float(x) if isinstance(x,np.floating) else int(x) if isinstance(x,np.integer) else str(x))

def clean():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

save_json(OUT/"config.json",{
    "V":V,"KS":KS,"SEEDS":SEEDS,"TRAIN_SIZES":TRAIN_SIZES,
    "N_VAL":N_VAL,"N_TEST":N_TEST,"device":str(DEVICE),
    "torch":torch.__version__,"python":sys.version
})


In [ ]:
def context_to_index(context,V):
    idx=torch.zeros(context.shape[:-1],dtype=torch.long,device=context.device)
    for j in range(context.shape[-1]): idx=idx*V+context[...,j]
    return idx

def generate_kernel(V,k,seed,device=DEVICE):
    g=torch.Generator(device=device); g.manual_seed(seed+10_000_019)
    logits=torch.randn((V**k,V),generator=g,device=device)
    return torch.softmax(logits,dim=-1)

def stationary_distribution(P,V,k,tol=1e-12,max_iter=100_000):
    """
    Numerically stable stationary distribution over the V^k shift-register states.

    Important:
    - move to CPU float64;
    - explicitly renormalize each kernel row (the saved/generated kernel originates
      in float32, so row sums are only approximately 1);
    - use a lazy transition 0.5 I + 0.5 T, which has the same stationary
      distribution and avoids periodicity/numerical oscillation;
    - renormalize probability mass each iteration.
    """
    Pnp = P.detach().double().cpu().numpy()
    Pnp = Pnp / Pnp.sum(axis=1, keepdims=True)

    n = V**k
    pi = np.full(n, 1.0/n, dtype=np.float64)

    states = np.arange(n, dtype=np.int64)
    suffix = states % (V**(k-1))
    next_idx = suffix[:,None] * V + np.arange(V, dtype=np.int64)[None,:]

    for it in range(max_iter):
        propagated = np.zeros_like(pi)
        np.add.at(
            propagated,
            next_idx.reshape(-1),
            (pi[:,None] * Pnp).reshape(-1)
        )
        propagated /= propagated.sum()

        new = 0.5*pi + 0.5*propagated
        new /= new.sum()

        err = float(np.max(np.abs(new-pi)))
        pi = new
        if err < tol:
            break
    else:
        raise RuntimeError(
            f"Stationary solver failed for k={k}; "
            f"last max-delta={err:.3e} after {max_iter} iterations"
        )

    # True (non-lazy) stationarity residual for audit.
    propagated = np.zeros_like(pi)
    np.add.at(
        propagated,
        next_idx.reshape(-1),
        (pi[:,None] * Pnp).reshape(-1)
    )
    propagated /= propagated.sum()
    residual = float(np.max(np.abs(propagated-pi)))

    return torch.from_numpy(pi), it+1, residual

@torch.no_grad()
def bayes_entropy(P,pi):
    P64=P.detach().double().cpu()
    P64=P64/P64.sum(dim=-1,keepdim=True)
    row_h=-(P64*torch.log(P64.clamp_min(1e-300))).sum(-1)
    return float((pi.double().cpu()*row_h).sum())

@torch.no_grad()
def generate_state_dataset(P,V,k,n,seed,warmup=256,batch_size=2048):
    g=torch.Generator(device=DEVICE); g.manual_seed(seed+7_000_009)
    B=min(batch_size,n)
    hist=torch.randint(0,V,(B,k),generator=g,device=DEVICE)
    for _ in range(warmup):
        probs=P[context_to_index(hist,V)]
        y=torch.multinomial(probs,1,generator=g).squeeze(1)
        hist=torch.cat([hist[:,1:],y[:,None]],1)
    states=[]; ys=[]; rem=n
    while rem>0:
        take=min(B,rem)
        s=hist[:take].clone()
        probs=P[context_to_index(s,V)]
        y=torch.multinomial(probs,1,generator=g).squeeze(1)
        states.append(s.cpu()); ys.append(y.cpu())
        probs_all=P[context_to_index(hist,V)]
        y_all=torch.multinomial(probs_all,1,generator=g).squeeze(1)
        hist=torch.cat([hist[:,1:],y_all[:,None]],1)
        rem-=take
    return torch.cat(states).long(),torch.cat(ys).long()

def state_onehot(states,V):
    return F.one_hot(states,num_classes=V).float().reshape(len(states),-1)


In [ ]:
def coverage_stats(states,pi,V,k):
    idx=context_to_index(states.to(DEVICE),V).cpu()
    counts=torch.bincount(idx,minlength=V**k)
    seen=counts>0
    unique=int(seen.sum()); total=V**k
    obs=counts[seen].float()
    return {
        "unique_states":unique,
        "total_states":total,
        "state_count_coverage":unique/total,
        "stationary_mass_coverage":float(pi.cpu()[seen].sum()),
        "median_visits_seen_state":float(obs.median()) if unique else 0.0,
        "min_visits_seen_state":int(obs.min()) if unique else 0,
        "max_visits_seen_state":int(obs.max()) if unique else 0,
    }


In [ ]:
class LinearNext(nn.Module):
    def __init__(self,d,V):
        super().__init__(); self.net=nn.Linear(d,V)
    def forward(self,x): return self.net(x)

class MLPNext(nn.Module):
    def __init__(self,d,V,width=512,depth=1):
        super().__init__()
        layers=[nn.Linear(d,width),nn.GELU()]
        if depth>=2: layers += [nn.Linear(width,width),nn.GELU()]
        layers += [nn.Linear(width,V)]
        self.net=nn.Sequential(*layers)
    def forward(self,x): return self.net(x)

PROBES={
    "linear":lambda d: LinearNext(d,V),
    "mlp":lambda d: MLPNext(d,V,512,1),
    "mlp_wide2":lambda d: MLPNext(d,V,1024,2),
}

def train_probe(model,Xtr,ytr,Xva,yva,epochs,lr):
    model=model.to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=WEIGHT_DECAY)
    tr=DataLoader(TensorDataset(Xtr,ytr),batch_size=PROBE_BATCH,shuffle=True)
    va=DataLoader(TensorDataset(Xva,yva),batch_size=PROBE_BATCH)
    best=float("inf"); best_state=None; bad=0
    for ep in range(epochs):
        model.train()
        for xb,yb in tr:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss=F.cross_entropy(model(xb),yb)
            loss.backward(); opt.step()
        model.eval(); tot=0.0; n=0
        with torch.no_grad():
            for xb,yb in va:
                xb,yb=xb.to(DEVICE),yb.to(DEVICE)
                loss=F.cross_entropy(model(xb),yb)
                tot += loss.item()*len(xb); n += len(xb)
        v=tot/n
        if v<best-1e-5:
            best=v
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            bad=0
        else:
            bad+=1
            if bad>=PATIENCE: break
    if best_state is not None: model.load_state_dict(best_state)
    return model

@torch.no_grad()
def eval_probe(model,X,y):
    dl=DataLoader(TensorDataset(X,y),batch_size=PROBE_BATCH)
    ce=0.0; cor=0; n=0
    model.eval()
    for xb,yb in dl:
        xb,yb=xb.to(DEVICE),yb.to(DEVICE)
        z=model(xb)
        ce += F.cross_entropy(z,yb).item()*len(xb)
        cor += (z.argmax(-1)==yb).sum().item()
        n += len(xb)
    return {"ce":ce/n,"acc":cor/n}


In [ ]:
@torch.no_grad()
def lookup_estimator_metrics(train_states,train_y,test_states,test_y,V,k,alpha=0.5):
    tr_idx=context_to_index(train_states.to(DEVICE),V).cpu()
    te_idx=context_to_index(test_states.to(DEVICE),V).cpu()
    counts=torch.full((V**k,V),float(alpha),dtype=torch.float64)
    flat=tr_idx*V+train_y
    binc=torch.bincount(flat,minlength=(V**k)*V).reshape(V**k,V).double()
    counts += binc
    probs=counts/counts.sum(-1,keepdim=True)
    p=probs[te_idx,test_y].clamp_min(1e-300)
    ce=float((-torch.log(p)).mean())
    pred=probs[te_idx].argmax(-1)
    acc=float((pred==test_y).float().mean())
    unseen=(binc.sum(-1)==0)
    test_unseen_rate=float(unseen[te_idx].float().mean())
    return {"lookup_ce":ce,"lookup_acc":acc,"test_samples_from_unseen_state_rate":test_unseen_rate,"alpha":alpha}


In [ ]:
raw_path=CSV_DIR/"A3b_true_state_coverage_raw.csv"

if raw_path.exists():
    raw=pd.read_csv(raw_path)
    print("Loaded existing:",raw_path)
else:
    rows=[]
    for k in KS:
        for seed in SEEDS:
            print(f"=== k={k} seed={seed} ===")
            set_seed(1000*k+seed)
            P=generate_kernel(V,k,seed)
            pi,n_iter,residual=stationary_distribution(P,V,k)
            H_star=bayes_entropy(P,pi)

            tr_states,tr_y=generate_state_dataset(P,V,k,max(TRAIN_SIZES),100_000+1000*k+seed)
            va_states,va_y=generate_state_dataset(P,V,k,N_VAL,200_000+1000*k+seed)
            te_states,te_y=generate_state_dataset(P,V,k,N_TEST,300_000+1000*k+seed)

            Xva=state_onehot(va_states,V)
            Xte=state_onehot(te_states,V)

            for n_train in TRAIN_SIZES:
                ckpt=RUN_DIR/f"k{k}_seed{seed}_n{n_train}.json"
                if ckpt.exists():
                    rows.extend(json.loads(ckpt.read_text())["rows"])
                    print(" loaded",ckpt.name)
                    continue

                s_tr=tr_states[:n_train]; y_tr=tr_y[:n_train]
                Xtr=state_onehot(s_tr,V)
                cov=coverage_stats(s_tr,pi,V,k)
                lookup=lookup_estimator_metrics(s_tr,y_tr,te_states,te_y,V,k,alpha=0.5)

                run_rows=[{
                    "k":k,"seed":seed,"n_train":n_train,"method":"lookup",
                    "H_star":H_star,"ce":lookup["lookup_ce"],
                    "excess_ce_over_Hstar":lookup["lookup_ce"]-H_star,
                    "acc":lookup["lookup_acc"],
                    "test_unseen_state_rate":lookup["test_samples_from_unseen_state_rate"],
                    **cov
                }]

                d=Xtr.shape[1]
                for probe_name,builder in PROBES.items():
                    print(f" n={n_train} probe={probe_name}")
                    epochs=LINEAR_EPOCHS if probe_name=="linear" else MLP_EPOCHS
                    lr=LINEAR_LR if probe_name=="linear" else MLP_LR
                    probe=train_probe(builder(d),Xtr,y_tr,Xva,va_y,epochs,lr)
                    met=eval_probe(probe,Xte,te_y)
                    run_rows.append({
                        "k":k,"seed":seed,"n_train":n_train,"method":probe_name,
                        "H_star":H_star,"ce":met["ce"],
                        "excess_ce_over_Hstar":met["ce"]-H_star,
                        "acc":met["acc"],
                        "test_unseen_state_rate":lookup["test_samples_from_unseen_state_rate"],
                        **cov
                    })
                    del probe; clean()

                save_json(ckpt,{"rows":run_rows})
                rows.extend(run_rows)
                del Xtr; clean()

            del P,pi,tr_states,tr_y,va_states,va_y,te_states,te_y,Xva,Xte
            clean()

    raw=pd.DataFrame(rows)
    raw.to_csv(raw_path,index=False)

display(raw.head(20))
print("Rows:",len(raw))


In [ ]:
summary=raw.groupby(["k","n_train","method"],as_index=False).agg(
    H_star_mean=("H_star","mean"),
    ce_mean=("ce","mean"),
    ce_sd=("ce","std"),
    excess_mean=("excess_ce_over_Hstar","mean"),
    excess_sd=("excess_ce_over_Hstar","std"),
    acc_mean=("acc","mean"),
    state_count_coverage_mean=("state_count_coverage","mean"),
    stationary_mass_coverage_mean=("stationary_mass_coverage","mean"),
    unseen_test_rate_mean=("test_unseen_state_rate","mean"),
    n_seeds=("seed","nunique"),
)
summary.to_csv(CSV_DIR/"A3b_true_state_coverage_summary.csv",index=False)
display(summary)

coverage_summary=raw.groupby(["k","n_train"],as_index=False).agg(
    state_count_coverage_mean=("state_count_coverage","mean"),
    state_count_coverage_sd=("state_count_coverage","std"),
    stationary_mass_coverage_mean=("stationary_mass_coverage","mean"),
    stationary_mass_coverage_sd=("stationary_mass_coverage","std"),
    unseen_test_rate_mean=("test_unseen_state_rate","mean"),
)
coverage_summary.to_csv(CSV_DIR/"A3b_coverage_summary.csv",index=False)
display(coverage_summary)


In [ ]:
fig,ax=plt.subplots(figsize=(7.5,4.8))
for k in KS:
    d=coverage_summary[coverage_summary.k==k].sort_values("n_train")
    ax.plot(d.n_train,d.stationary_mass_coverage_mean,marker="o",label=f"k={k}")
ax.set_xscale("log")
ax.set_xlabel("Training samples")
ax.set_ylabel("Stationary probability mass covered")
ax.set_title("True-state training coverage")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR/"A3b_stationary_mass_coverage.png",dpi=300)
plt.show()

fig,ax=plt.subplots(figsize=(8,5))
for (k,method),d in summary.groupby(["k","method"]):
    d=d.sort_values("n_train")
    ax.plot(d.n_train,d.excess_mean,marker="o",label=f"k={k} {method}")
ax.axhline(0,linewidth=1)
ax.set_xscale("log")
ax.set_xlabel("Training samples")
ax.set_ylabel("CE - H* (nats)")
ax.set_title("True-state readout gap vs coverage and probe capacity")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR/"A3b_excess_ce_vs_data.png",dpi=300)
plt.show()


In [ ]:
expected=len(KS)*len(SEEDS)*len(TRAIN_SIZES)*(1+len(PROBES))
assert len(raw)==expected,(len(raw),expected)
assert set(raw["method"])=={"lookup","linear","mlp","mlp_wide2"}
assert np.isfinite(raw.select_dtypes(include=[np.number]).to_numpy()).all()

audit={
    "status":"PASS",
    "rows":len(raw),
    "k_values":KS,
    "seeds":SEEDS,
    "train_sizes":TRAIN_SIZES,
    "methods":sorted(raw["method"].unique().tolist()),
    "all_numeric_finite":True,
}
save_json(OUT/"A3b_audit.json",audit)
print(json.dumps(audit,indent=2))

(OUT/"README.txt").write_text(
    "Reviewer A3b outputs\n\n"
    "csv/A3b_true_state_coverage_raw.csv\n"
    "csv/A3b_true_state_coverage_summary.csv\n"
    "csv/A3b_coverage_summary.csv\n"
    "figures/A3b_stationary_mass_coverage.png\n"
    "figures/A3b_excess_ce_vs_data.png\n"
    "A3b_audit.json\n",
    encoding="utf-8"
)

zip_path=Path.cwd()/"Reviewer_A3b_TrueState_Coverage_RESULTS.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for f in OUT.rglob("*"):
        if f.is_file():
            z.write(f,f.relative_to(OUT.parent))
print("Created:",zip_path)
